# Predicting Next-Week Sales of Clothing Items

INFO 1998 final project

**Team:** Cerisa Katchmart (ck865) and Lara De (ld569)

## Hypothesis
A product's recent sales, age, price changes and attributes together predict its next-week sales
more accurately than its recent sales alone.

## Research question
Is a predictive model better at forecasting next-week sales than just basic trend baselines alone?

**Problem type:** regression (we predict a number: units sold next week).

## 1. Setup

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

sys.path.append("..")  # so we can import from src/
from src.data import build_weekly_panel, make_synthetic_panel

pd.set_option("display.max_columns", 30)

## 2. Load the data

We use the [H&M dataset from Kaggle](https://www.kaggle.com/competitions/h-and-m-personalized-fashion-recommendations/overview).
The raw file is huge, so run `python -m src.prepare_hm` once (see the README) to make a smaller
weekly file. If that file isn't there yet, this notebook falls back to **fake demo data** so you can
see it run. **Do not report results from demo data.**

In [ ]:
TX_PATH = "../data/processed/hm_weekly_transactions.csv"
ITEMS_PATH = "../data/processed/hm_items.csv"

if os.path.exists(TX_PATH):
    transactions = pd.read_csv(TX_PATH)
    items = pd.read_csv(ITEMS_PATH)
    panel = build_weekly_panel(transactions, items)
    USING_DEMO = False
else:
    panel = make_synthetic_panel()
    USING_DEMO = True
    print("WARNING: using FAKE demo data. Do not report these results.")

# one row per item per week
panel.head()

## 3. Explore the data

In [ ]:
print("rows:", len(panel))
print("items:", panel["item_id"].nunique())
print("weeks:", panel["week"].min().date(), "to", panel["week"].max().date())
print("missing values:")
print(panel.isna().sum())
panel[["units", "price"]].describe()

In [ ]:
# How many units does a typical item sell in a week?
plt.hist(panel["units"].clip(upper=panel["units"].quantile(0.99)), bins=40)
plt.xlabel("units sold in a week (clipped at the 99th percentile)")
plt.ylabel("number of item-weeks")
plt.title("Most item-weeks have few sales")
plt.show()

In [ ]:
# Total sales over time
panel.groupby("week")["units"].sum().plot(figsize=(9, 3), title="Total units sold per week")
plt.ylabel("units")
plt.show()

## 4. Feature engineering

Each row will describe one item in one week. We predict the **next** week's units.
Every feature may only use information we would know at the end of that week, otherwise we
"cheat" by peeking into the future (this is called **data leakage**).

| feature | meaning |
|---|---|
| `lag_1` ... `lag_4` | units sold this week, 1 week ago, 2 weeks ago, 3 weeks ago |
| `roll_mean_4` | average units over the last 4 weeks |
| `price` | this week's price |
| `price_change` | % change in price vs. last week |
| `age_weeks` | weeks since the item's first sale |
| `week_sin`, `week_cos` | time of year (so the model can learn seasons) |
| attribute columns | product group, colour, etc. (one-hot encoded) |

In [ ]:
df = panel.sort_values(["item_id", "week"]).copy()
by_item = df.groupby("item_id")

# --- recent sales ---
df["lag_1"] = df["units"]                       # this week
df["lag_2"] = by_item["units"].shift(1)
df["lag_3"] = by_item["units"].shift(2)
df["lag_4"] = by_item["units"].shift(3)
for col in ["lag_2", "lag_3", "lag_4"]:
    df[col] = df[col].fillna(df["lag_1"])       # new items: not enough history yet
df["roll_mean_4"] = by_item["units"].transform(lambda s: s.rolling(4, min_periods=1).mean())

# --- price, age, season ---
df["price_change"] = by_item["price"].pct_change().fillna(0)
df["age_weeks"] = (df["week"] - df["launch_week"]).dt.days // 7
week_of_year = df["week"].dt.isocalendar().week.astype(int)
df["week_sin"] = np.sin(2 * np.pi * week_of_year / 52)
df["week_cos"] = np.cos(2 * np.pi * week_of_year / 52)

# --- what we want to predict: NEXT week's units ---
df["target"] = by_item["units"].shift(-1)
df = df.dropna(subset=["target"])               # each item's final week has no "next week"

df[["item_id", "week", "lag_1", "roll_mean_4", "price", "price_change", "age_weeks", "target"]].head()

In [ ]:
# Attribute columns (anything that isn't one of our own columns) -> one-hot encoding.
# For H&M we skip very detailed columns so the model doesn't get hundreds of dummy columns.
base_cols = {"item_id", "week", "units", "price", "launch_week", "lag_1", "lag_2", "lag_3", "lag_4",
             "roll_mean_4", "price_change", "age_weeks", "week_sin", "week_cos", "target"}
attribute_cols = [c for c in df.columns if c not in base_cols and c != "product_type_name"]
print("attribute columns:", attribute_cols)

df = pd.get_dummies(df, columns=attribute_cols, dtype=int)
dummy_cols = [c for c in df.columns if any(c.startswith(a + "_") for a in attribute_cols)]
print("number of one-hot columns:", len(dummy_cols))

## 5. Visualize relationships

Do our features seem related to next week's sales?

In [ ]:
sample = df.sample(min(5000, len(df)), random_state=0)
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
axes[0].scatter(sample["lag_1"], sample["target"], s=4, alpha=0.4)
axes[0].set_xlabel("this week's units"); axes[0].set_ylabel("next week's units")
axes[1].scatter(sample["price_change"], sample["target"], s=4, alpha=0.4)
axes[1].set_xlabel("price change vs. last week")
axes[2].scatter(sample["age_weeks"], sample["target"], s=4, alpha=0.4)
axes[2].set_xlabel("age in weeks")
plt.tight_layout()
plt.show()

## 6. Train / test split

Sales are a time series, so we **can't shuffle** rows randomly (that would let the model train on the
future). Instead we train on earlier weeks and test on the **last 10 weeks**. We leave a one-week gap
so the last training row's target doesn't overlap the test period.

In [ ]:
TEST_WEEKS = 10
weeks = np.sort(df["week"].unique())
cutoff = weeks[-TEST_WEEKS]

train = df[df["week"] < cutoff - np.timedelta64(7, "D")]
test = df[df["week"] >= cutoff]
print("train rows:", len(train), " test rows:", len(test))

## 7. Baselines

Before using machine learning, how well do two simple rules do?

1. **Last week:** predict next week = this week.
2. **4-week average:** predict next week = average of the last 4 weeks.

If our model can't beat these, it isn't worth using.

In [ ]:
def mae(y, p):
    return np.mean(np.abs(y - p))

def rmse(y, p):
    return np.sqrt(np.mean((y - p) ** 2))

y_test = test["target"]
results = {}
results["baseline: last week"] = test["lag_1"]
results["baseline: 4-week avg"] = test["roll_mean_4"]

## 8. Models

We compare **two versions of the same model** (linear regression):

- **Sales only:** uses just recent sales (`lag_1`..`lag_4`, `roll_mean_4`).
- **Full:** also uses price, price change, age, season and attributes.

Because only the features differ, any difference in accuracy comes from the extra features, which is
exactly what our hypothesis is about.

In [ ]:
sales_features = ["lag_1", "lag_2", "lag_3", "lag_4", "roll_mean_4"]
full_features = sales_features + ["price", "price_change", "age_weeks", "week_sin", "week_cos"] + dummy_cols

def fit_and_predict(model, features):
    model.fit(train[features], train["target"])
    return np.clip(model.predict(test[features]), 0, None)  # sales can't be negative

results["linear: sales only"] = fit_and_predict(LinearRegression(), sales_features)
results["linear: full"] = fit_and_predict(LinearRegression(), full_features)

table = pd.DataFrame({
    name: {"MAE": mae(y_test, p), "RMSE": rmse(y_test, p)} for name, p in results.items()
}).T.round(3)
table

**MAE** (mean absolute error) is the average number of units we're off by each week. **RMSE** is similar
but punishes big misses more. Lower is better for both.

## 9. A second model type: random forest

A linear model can only learn straight-line relationships. A random forest can learn things like
"a price drop only helps for new items". Same two feature sets again.

In [ ]:
def forest():
    return RandomForestRegressor(n_estimators=100, max_depth=8, min_samples_leaf=20,
                                 n_jobs=-1, random_state=0)

results["forest: sales only"] = fit_and_predict(forest(), sales_features)
results["forest: full"] = fit_and_predict(forest(), full_features)

table = pd.DataFrame({
    name: {"MAE": mae(y_test, p), "RMSE": rmse(y_test, p)} for name, p in results.items()
}).T.round(3)
table

## 10. Cross-validation

One test period could be lucky or unlucky. Here we repeat the experiment 4 times, each time training on
all earlier weeks and testing on the next 6-week block (an "expanding window", the time-series version of
cross-validation).

In [ ]:
def cv_mae(make_model, features, n_folds=4, fold_weeks=6):
    scores = []
    for k in range(n_folds):
        end = len(weeks) - (n_folds - 1 - k) * fold_weeks
        start = end - fold_weeks
        tr = df[df["week"] < weeks[start] - np.timedelta64(7, "D")]
        te = df[df["week"].isin(weeks[start:end])]
        model = make_model()
        model.fit(tr[features], tr["target"])
        pred = np.clip(model.predict(te[features]), 0, None)
        scores.append(mae(te["target"], pred))
    return scores

cv = {
    "linear: sales only": cv_mae(LinearRegression, sales_features),
    "linear: full": cv_mae(LinearRegression, full_features),
    "forest: sales only": cv_mae(forest, sales_features),
    "forest: full": cv_mae(forest, full_features),
}
cv_table = pd.DataFrame(cv, index=[f"fold {i+1}" for i in range(4)]).T
cv_table["average"] = cv_table.mean(axis=1)
cv_table.round(3)

## 11. Results

In [ ]:
final = pd.Series({name: mae(y_test, p) for name, p in results.items()})
ax = final.sort_values().plot.barh(figsize=(7, 4), color="steelblue")
ax.set_xlabel("average error in units per item-week (lower is better)")
ax.set_title("Test-set error by method")
plt.tight_layout()
plt.show()

In [ ]:
# Which features did the linear model rely on most? (coefficients; features are on different scales,
# so be careful comparing sizes, they only show direction and rough importance)
lin = LinearRegression().fit(train[full_features], train["target"])
coefs = pd.Series(lin.coef_, index=full_features).sort_values()
pd.concat([coefs.head(5), coefs.tail(5)])

In [ ]:
# Actual vs predicted for the best model on the test set
best_name = final.idxmin()
plt.figure(figsize=(5, 5))
plt.scatter(y_test, results[best_name], s=4, alpha=0.3)
lim = float(np.nanpercentile(y_test, 99))
plt.plot([0, lim], [0, lim], "r--")
plt.xlim(0, lim); plt.ylim(0, lim)
plt.xlabel("actual next-week units"); plt.ylabel("predicted")
plt.title(f"{best_name}")
plt.show()

## 12. Conclusion (write this part yourselves)

Answer these in your own words, using the numbers above:

1. **Baselines:** How accurate were the "last week" and "4-week average" rules (MAE)?
2. **Research question:** Did the best model beat the best baseline? By how much, in units per item-week?
   Is that difference meaningful for a store?
3. **Hypothesis:** Did adding price, age, season and attributes (full) beat sales only? Was it true for
   both model types and across the cross-validation folds, or only some?
4. **Why?** Which features seem to matter? Does that make sense for clothing?
5. **Limitations:** We only used the top-selling items, dropped items that were already on sale when the
   data starts, and H&M's price is a scaled number. How might each affect the results? What would you try
   next (more features such as promotions, holidays, or per-item models)?

> If this notebook is still running on demo data, the results above are meaningless. Re-run it after
> preparing the real H&M files.